In [15]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

In [16]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [17]:
llm = ChatOpenAI()
def chat_node(state: ChatState):
    messages = state['messages']
    response = llm.invoke(messages)
    return {'messages': [response]}


In [18]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

#add nodes
graph.add_node('chat_node', chat_node)
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [22]:
initial_state = {
    'messages': [HumanMessage(content='What is the capital of India')]
}
chatbot.invoke(initial_state)['messages'][-1].content

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [24]:
thread_id = '1'
while True: 
    user_message = input('Type here: ')
    print('User:' , user_message)
    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break
    config = {'configurable': {'thread_id': thread_id}}
    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]}, config=config)
    print('AI', response['messages'][-1].content)

User: Hi, I am Neha
AI Hello Neha! How can I assist you today?
User: whats my name
AI Your name is Neha.
User: what is 1_1
AI 1_1 is equal to 0.
User: 1+1
AI 1 + 1 is equal to 2.
User: add 5 to the previous result
AI Adding 5 to the previous result of 2 (1 + 1), we get: 2 + 5 = 7.
User: my name is
AI Your name is Neha.
User: exit


In [25]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='Hi, I am Neha', additional_kwargs={}, response_metadata={}, id='fd720189-1277-4362-a58a-dded6fdce4ab'), AIMessage(content='Hello Neha! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 13, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Cv0n5crmeYia3wZTsQBxvlDADDZg0', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b935a-f4bd-78f0-97d8-23948fe120ae-0', usage_metadata={'input_tokens': 13, 'output_tokens': 11, 'total_tokens': 24, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'r